# 01 · 共享 global R1 的真实层 G1

逐层采集 q/up 输入侧和 down 输出侧；所有面板共享同一个 Q。完整 reduction groups 与输出 hidden 轴保留。默认只执行 preflight，不下载或启动模型。

GitHub / VS Code 工作流先运行 `00_Colab_GitHub_Workspace.ipynb`；详见 [操作说明](../docs/COLAB_VSCODE.md)。已有 ZIP 入口仍可用。

## 路径与环境配置

GitHub / VS Code 首先运行 `00_Colab_GitHub_Workspace.ipynb`；本 notebook 会识别 `/content/CellQuant` 并读取运行时路径配置。
已有 ZIP 仍可设置 `KIT_ARCHIVE` 和 `EXTRACT_ARCHIVE=True` 解压，或直接指定 `PROJECT_ROOT`。
默认 smoke / manifest-only，不下载模型、不启动全量 sweep。基础依赖缺失时才设置 `INSTALL_BASE_DEPENDENCIES=True`，不替换宿主 torch。
真实模型授权通过环境变量或隐藏输入提供，不能写入 notebook 输出或 Git；阶段结束用 workspace backup 保存到 Drive。


In [1]:
from pathlib import Path
import os, sys, json

# Colab: Git clone 到 /content/CellQuant；也兼容完整 ZIP 解压目录。
# AutoDL: 建议 /root/autodl-tmp/cellquant_research_kit。
# 留 None 时依次查找当前目录、父目录和上述两个位置。
PROJECT_ROOT = os.environ.get("CELLQUANT_PROJECT_ROOT") or ("/content/CellQuant" if Path("/content/CellQuant/pyproject.toml").is_file() else None)
RUN_ROOT = os.environ.get("CELLQUANT_RUN_ROOT") or None
KIT_ARCHIVE = None  # 例：/content/drive/MyDrive/cellquant_research_kit.zip
EXTRACT_ARCHIVE = False  # 首次运行且 ZIP 已上传时才改 True
ARCHIVE_DESTINATION = "/content/cellquant_workspace" if Path("/content").exists() else str(Path.cwd().parent / "cellquant_workspace")
MOUNT_DRIVE = False
DURABLE_ROOT = None  # 例：/content/drive/MyDrive/cellquant_runs；不存权重或 token
INSTALL_BASE_DEPENDENCIES = False
SEED = 20261115

In [2]:
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Drive mounting skipped; outputs remain under RUN_ROOT until copied.")

Drive mounting skipped; outputs remain under RUN_ROOT until copied.


In [3]:
archive_roots = []
if EXTRACT_ARCHIVE:
    import zipfile, hashlib
    if not KIT_ARCHIVE or not Path(KIT_ARCHIVE).is_file():
        raise FileNotFoundError("先上传完整 ZIP 并设置 KIT_ARCHIVE")
    archive_path = Path(KIT_ARCHIVE)
    destination = Path(ARCHIVE_DESTINATION).resolve()
    digest = hashlib.sha256(archive_path.read_bytes()).hexdigest()
    receipt = destination / "archive_source_sha256.txt"
    if destination.exists() and any(destination.iterdir()):
        if not receipt.is_file() or receipt.read_text().strip() != digest:
            raise FileExistsError("目标目录已有不同内容；请设置新的 ARCHIVE_DESTINATION，避免覆盖修改")
        print("Same archive already extracted; preserving existing project edits.")
    else:
        with zipfile.ZipFile(archive_path) as archive:
            infos = archive.infolist()
            if sum(info.file_size for info in infos) > 2 * 2**30:
                raise ValueError("项目包解压体积超过 2GiB；请勿把基础权重放进代码包")
            for info in infos:
                target = (destination / info.filename).resolve()
                if not target.is_relative_to(destination) or (info.external_attr >> 16) & 0o170000 == 0o120000:
                    raise ValueError("Unsafe archive member")
            destination.mkdir(parents=True, exist_ok=True)
            archive.extractall(destination)
        receipt.write_text(digest + "\n")
    archive_roots = [destination, destination / "cellquant_research_kit"]

candidates = ([Path(PROJECT_ROOT)] if PROJECT_ROOT else []) + archive_roots + [
    Path.cwd(), Path.cwd().parent,
    Path("/content/cellquant_research_kit"),
    Path("/root/autodl-tmp/cellquant_research_kit"),
]
roots = [p.resolve() for p in candidates
         if (p / "pyproject.toml").is_file() and (p / "src/cellquant").is_dir()]
if not roots:
    raise FileNotFoundError("请先解压完整项目包，并设置 PROJECT_ROOT；只上传 notebook 不足以运行。")
PROJECT_ROOT = roots[0]
sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.environ["CELLQUANT_PROJECT_ROOT"] = str(PROJECT_ROOT)

if INSTALL_BASE_DEPENDENCIES:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r",
                           str(PROJECT_ROOT / "requirements/base.txt")])

from cellquant.runtime import (atomic_json, environment_report, resolve_paths,
                              build_manifest, verify_manifest, copy_completed_run)
PATHS = resolve_paths(PROJECT_ROOT, RUN_ROOT, DURABLE_ROOT)
print("Project:", PATHS["project"])
print("Runs:", PATHS["runs"])

os.chdir(PROJECT_ROOT)
os.environ["PYTHONPATH"] = str(PROJECT_ROOT / "src") + os.pathsep + os.environ.get("PYTHONPATH", "")
os.environ.setdefault("HF_HOME", str(PROJECT_ROOT / ".cache/huggingface"))
os.environ.setdefault("PYTHONUTF8", "1")

# Separate notebook kernels reuse the same local runtime path configuration.
workspace_record = PROJECT_ROOT / ".cache/colab_workspace.json"
if workspace_record.is_file():
    from cellquant.workspace import configure_workspace
    saved = json.loads(workspace_record.read_text(encoding="utf-8"))
    configure_workspace(PROJECT_ROOT, saved["environment"]["CELLQUANT_DRIVE_ROOT"])


Project: D:\python_project\quant-llm
Runs: D:\python_project\quant-llm\runs


先显式准备：`python scripts/prepare_spinquant.py`；`python scripts/prepare_experiment.py --config configs/colab_pilot.json --output runs/prepared/g1`。设置 HF_TOKEN 环境变量或 Colab Secrets，禁止打印。T4 必须记录 FP16；本实现需要足够 CPU RAM 暂存权重和完整面板，高 RAM Colab 才可能满足。

In [4]:
from cellquant.experiment import run_experiment
from cellquant.runtime import stream_command, read_json
from cellquant.gates import evaluate_file
OUT = PATHS["runs"] / "01_pilot"
OUT.mkdir(parents=True, exist_ok=True)
RUN_REAL_MODEL = False
PILOT_CONFIG_PATH = Path(os.environ.get("CELLQUANT_PILOT_CONFIG", PROJECT_ROOT / "runs/prepared/g1/config.json"))
if not PILOT_CONFIG_PATH.is_file():
    PILOT_CONFIG_PATH = PROJECT_ROOT / "configs/colab_pilot.json"
preflight = run_experiment(PILOT_CONFIG_PATH, project_root=PROJECT_ROOT, preflight_only=True)
print(preflight["readiness"])


{'status': 'BLOCKED', 'missing': ['Missing immutable model_revision; run scripts/prepare_experiment.py', 'Missing immutable tokenizer_revision; run scripts/prepare_experiment.py', 'Missing immutable dataset_revision; run scripts/prepare_experiment.py', 'Pinned model shard size/SHA-256 metadata missing', 'Missing real fit/dev token manifest for seed 11', 'Missing real fit/dev token manifest for seed 23', 'Available CPU RAM 1.59 GiB below 24 GiB model/panel staging requirement', 'Record an explicit actual model dtype; no silent fallback'], 'environment': {'schema_version': '1.0', 'created_at_utc': '2026-11-16T07:33:21.396438+00:00', 'python': '3.11.18', 'executable': 'D:\\python_project\\quant-llm\\.venv\\Scripts\\python.exe', 'platform': 'Windows-10-10.0.26200-SP0', 'machine': 'AMD64', 'packages': {'numpy': '2.4.6', 'scipy': '1.17.1', 'torch': '2.6.0+cu118', 'transformers': '4.44.2', 'datasets': '2.20.0', 'accelerate': '0.34.2', 'safetensors': '0.8.0', 'nbformat': '5.11.1', 'nbclient': 

In [5]:
if RUN_REAL_MODEL:
    if preflight["readiness"]["status"] != "READY":
        raise RuntimeError("BLOCKED: resolve the recorded prerequisites; RUN_REAL_MODEL is not gate evidence")
    process = stream_command([sys.executable, "-m", "cellquant.experiment", "--config", str(PILOT_CONFIG_PATH)],
                             OUT / "pilot.log", cwd=PROJECT_ROOT, timeout_s=5*3600)
    if process["returncode"] != 0:
        raise RuntimeError(f"Real pilot incomplete; inspect {OUT / 'pilot.log'}")
else:
    print("NOT_RUN: real-model execution switch is false")


NOT_RUN: real-model execution switch is false


真实作业写入唯一 `runs/experiments/<run_id>`。把该目录中的 G1_evidence.json 设为下方路径；G1 从原始测量重算。运行标志不会更改门槛。超事件/时间预算记录 RESOURCE_BLOCKED；未完成控制组不产生性能结论。

In [6]:
G1_EVIDENCE = Path(os.environ.get("CELLQUANT_G1_EVIDENCE", PROJECT_ROOT / "runs/gates/G1_evidence.json"))
evidence_path = G1_EVIDENCE if G1_EVIDENCE.is_file() else PROJECT_ROOT / "examples/G1_evidence_NOT_RUN.json"
gate = evaluate_file("G1", evidence_path)
atomic_json(OUT / "G1.json", gate)
print(gate["status"], gate["missing_fields"])


INSUFFICIENT_EVIDENCE ['source_type', 'provenance', 'artifact_refs', 'metrics', 'metrics.g0_pass', 'metrics.real_panel_provenance_verified', 'metrics.real_model_parameter_billions', 'metrics.calibration_seed_count', 'metrics.layer_count', 'metrics.site_count', 'metrics.fit_sequence_count', 'metrics.heldout_sequence_count', 'metrics.calibration_heldout_disjoint', 'metrics.shared_global_r1', 'metrics.full_quant_groups_preserved', 'metrics.down_projection_output_side_correct', 'metrics.matched_solver_controls_complete', 'metrics.same_hardware_timing', 'metrics.each_seed_pooled_heldout_improved', 'metrics.max_site_nmse_regression_vs_best_pct', 'metrics.median_nmse_gain_vs_hadamard_pct', 'metrics.median_nmse_gain_vs_best_matched_solver_pct', 'metrics.site_win_fraction_vs_best_matched_solver', 'metrics.median_nmse_regression_vs_best_matched_solver_pct', 'metrics.site_noninferior_fraction_at_2pct', 'metrics.pilot_calibration_speedup']
